하이브리드 리랭킹과 재측정

In [1]:
import os  # 작업 디렉터리 변경(os.chdir)에 사용
import pathlib  # 경로를 객체로 다루기 위한 모듈
import sys  # 모듈 검색 경로(sys.path)를 조작하기 위해 사용

here = pathlib.Path.cwd()  # 현재 작업 디렉터리 (노트북을 처음 열면 sandbox/W7/Thu)
ROOT = here.parents[2] if here.name == "Thu" else here  # Thu 폴더면 세 단계 위(프로젝트 루트), 이미 루트로 옮겨졌으면 그대로 사용
os.chdir(ROOT)  # 이후 상대 경로가 프로젝트 루트 기준이 되도록 작업 디렉터리 이동
SANDBOX = ROOT / "sandbox" / "w7" / "Thu"  # 오늘 실습 폴더 경로

print("프로젝트 루트  :", ROOT)  # 루트를 제대로 잡았는지 눈으로 확인

if str(ROOT / 'backend') not in sys.path:  # backend 폴더가 아직 import 경로에 없으면
    sys.path.insert(0, str(ROOT / 'backend'))  # 맨 앞에 추가 → `import app...` 가 가능해짐 (커널 재시작 시 사라지므로 다시 실행)

SAMPLES = ROOT / "sandbox" / "w5" / "Mon" / "samples"  # W5 월요일에 만든 샘플 문서 폴더
FORMATS = SAMPLES / "_formats"  # 포맷별 샘플 파일이 모인 하위 폴더

프로젝트 루트  : d:\hanwha-agent


In [2]:
from sqlalchemy import func, inspect, select  # func: COUNT 등 SQL 함수, inspect: DB 구조 조회, select: SELECT 문 생성
from app.db.session import get_engine, session_scope  # 엔진(DB 연결) 생성 함수와 트랜잭션 세션 컨텍스트 매니저
from app.models.document import Chunk  # chunks 표의 ORM 모델
from app.models.eval import EvalRun, EvalWeakType  # 평가 실행 기록(eval_runs)·오답 유형(eval_weak_types) 모델
SKIP = ('.git', '.pytest_cache', '.venv', '__pycache__', 'sandbox')  # 파일 개수를 셀 때 제외할 폴더 이름들
files = [p for p in ROOT.rglob('*') if p.is_file() and (not any((s in p.parts for s in SKIP)))]  # 루트 아래 모든 파일 중 제외 폴더에 속하지 않은 것만 모음
migrations = sorted((ROOT / 'backend/app/db/migrations/versions').glob('*.py'))  # Alembic 마이그레이션 파일 목록
table_names = inspect(get_engine()).get_table_names()  # 실제 DB에 존재하는 표 이름 목록
print(f'프로젝트 파일 : {len(files)}개   (.git · .pytest_cache · .venv · __pycache__ · sandbox 제외)')  # 프로젝트 파일 수 출력
print(f'chunks 컬럼   : {len(Chunk.__table__.columns)}        ← 어제와 같습니다. 오늘도 칸을 안 늘립니다')  # chunks 표의 컬럼 수 출력
print(f'DB 표         : {len(table_names)}       ← 어제 저녁에 9 에서 11 이 됐습니다')  # DB 표 개수 출력
print(f'마이그레이션  : {len(migrations)}건      ← 오늘은 0건이 늘 예정입니다')  # 마이그레이션 파일 수 출력
if 'eval_runs' not in table_names:  # 평가 결과 표가 아직 없으면
    print('eval_runs        : 표가 없습니다 — 아래 복원 셀이 만듭니다')  # 안내만 하고 조회는 건너뜀
else:  # 표가 있으면 행 수를 조회
    with session_scope() as s:  # 세션을 열고, 블록이 끝나면 커밋/종료 (에러 시 롤백)
        runs = s.scalar(select(func.count()).select_from(EvalRun))  # SELECT COUNT(*) FROM eval_runs
        weak = s.scalar(select(func.count()).select_from(EvalWeakType))  # SELECT COUNT(*) FROM eval_weak_types
    print(f'eval_runs        : {runs}행   ← 어제 「기본 검색」 한 줄')  # 평가 실행 기록 수 출력
    print(f'eval_weak_types  : {weak}행   ← 어제 저녁의 오답 유형 넷')  # 오답 유형 수 출력


프로젝트 파일 : 174개   (.git · .pytest_cache · .venv · __pycache__ · sandbox 제외)
chunks 컬럼   : 7        ← 어제와 같습니다. 오늘도 칸을 안 늘립니다
DB 표         : 11       ← 어제 저녁에 9 에서 11 이 됐습니다
마이그레이션  : 8건      ← 오늘은 0건이 늘 예정입니다
eval_runs        : 1행   ← 어제 「기본 검색」 한 줄
eval_weak_types  : 4행   ← 어제 저녁의 오답 유형 넷


In [3]:
from sqlalchemy import select  # SELECT 문 생성 함수
from app.db.session import session_scope  # 트랜잭션 세션 컨텍스트 매니저
from app.models.eval import EvalWeakType  # 오답 유형(eval_weak_types) ORM 모델

with session_scope() as s:  # 세션을 열고 블록이 끝나면 자동으로 정리
    weak_rows = [(w.ord, w.label, w.count, w.accuracy, w.diagnosis) for w in s.scalars(select(EvalWeakType).order_by(EvalWeakType.ord))]  # 오답 유형을 순서(ord)대로 읽어 필요한 칸만 튜플로 꺼냄 (세션이 닫히기 전에 값 복사)

if not weak_rows:  # 저장된 오답 유형이 하나도 없으면
    print('어제 남긴 오답 유형이 없습니다 — 지난시간에 테스트를 돌리지 않았거나 DB 를 새로 만든 것입니다.')  # 원인 안내
for (ord_, label, cnt, acc, diag) in weak_rows:  # 각 오답 유형을 하나씩 풀어서
    print(f'[{ord_ + 1}] {label:<18}{cnt:>3}건   정답률 {acc:>5}   ← {diag}')  # 번호(1부터)·이름·건수·정답률·진단을 정렬해 출력

[1] 타 부서 문서 참조          4건   정답률  50.0   ← 보안등급 정상 차단 · 시험지 오류 2건
[2] 규정에 없는 사항           6건   정답률     —   ← 근거부족 정상
[3] 표 안의 조건 조합          6건   정답률  83.3   ← 표가 조항 단위로 잘림
[4] 단일 조항              10건   정답률  90.0   ← 정답 문서가 상위에 없음


In [4]:
import json  # 리포트 JSON 파일을 읽기 위해 사용

REPORTS = ROOT / 'evals' / 'reports'  # 평가 리포트가 저장되는 폴더
saved = sorted(REPORTS.glob('eval_*.json'))  # 리포트 파일 목록 (파일명에 시각이 들어가므로 정렬하면 마지막이 최신)
wrong = []  # 틀린 문항을 담을 리스트 (리포트가 없어도 이후 셀에서 쓸 수 있게 미리 정의)

if not saved:  # 리포트 파일이 하나도 없으면
    print('어제 리포트가 없습니다 — 터미널에서 run_evals 명령을 --no-save 없이 한 번 더 치면 파일이 떨어집니다.')  # 만드는 방법 안내
else:  # 리포트가 있으면
    latest = saved[-1]  # 가장 최근 리포트
    report = json.loads(latest.read_text(encoding='utf-8'))  # 파일을 읽어 dict로 변환
    wrong = [r for r in report['runs'][0]['results'] if r['ok'] is False]  # 첫 번째 실행 결과 중 오답(ok=False)만 추림
    print(f'어제 리포트 : {latest.relative_to(ROOT).as_posix()}')  # 루트 기준 상대 경로로 파일 위치 출력
    print(f'틀린 문항 {len(wrong)}개 중 셋만 봅니다')  # 오답 개수 출력
    for r in wrong[:3]:  # 앞의 세 개만
        print(f"  {r['id']}  {r['type']:<12} ← {r['why']}")  # 문항 id·유형·틀린 이유 출력


어제 리포트 : evals/reports/eval_20261012_113133.json
틀린 문항 6개 중 셋만 봅니다
  G07  단일 조항        ← 정답 문서가 상위에 없음
  G16  표 안의 조건 조합   ← 정답 문서가 상위에 없음
  G23  타 부서 문서 참조   ← 정답 문서가 상위에 없음


In [5]:
from sqlalchemy import func, select
from app.db.session import session_scope
from app.models.document import Chunk
WORDS = ['출장비', '여비', '숙박', '1박당', '1박 당']
with session_scope() as s:
    for word in WORDS:
        n = s.scalar(select(func.count()).select_from(Chunk).where(Chunk.text.like(f'%{word}%')))
        print(f"LIKE '%{word}%'".ljust(16), '→', f'{n}건')

LIKE '%출장비%'     → 0건
LIKE '%여비%'      → 33건
LIKE '%숙박%'      → 27건
LIKE '%1박당%'     → 2건
LIKE '%1박 당%'    → 0건


In [6]:
from app.rag.retriever import expand_queries

for question in ["출장비 얼마까지 나오나요?", "프로젝트 착수 단계 산출물은?"]:
    queries = expand_queries(question, 3)
    print(f'"{question}"  → {len(queries)}갈래')
    for i, one in enumerate(queries, 1):
        print(f"    [{i}] {one}")

"출장비 얼마까지 나오나요?"  → 2갈래
    [1] 출장비 얼마까지 나오나요?
    [2] 출장비 얼마까지 나오나요? 여비 일비 숙박비 식비
"프로젝트 착수 단계 산출물은?"  → 1갈래
    [1] 프로젝트 착수 단계 산출물은?


In [7]:
import collections
import yaml

items = yaml.safe_load((ROOT / 'evals' / 'golden_set.yaml').read_text(encoding='utf-8'))['items']
dist = collections.Counter()
picked = {}

for item in items:
    queries = expand_queries(item['question'], 3)
    dist[len(queries)] += 1
    picked.setdefault(len(queries), (item, queries))

total = sum(dist.values())
average = sum((k * v for (k, v) in dist.items())) / total
print(f'갈래 수 분포 : {dict(sorted(dist.items()))}   총 {total}질의 · 평균 {average:.2f}개')

for n in (1, 3):
    if n not in picked:
        continue
    (item, queries) = picked[n]
    print(f'''{n}갈래 예 : {item['id']}  "{item['question']}"''')
    for one in queries:
        print(f'             └ "{one}"')

갈래 수 분포 : {1: 14, 2: 11, 3: 5}   총 30질의 · 평균 1.70개
1갈래 예 : G01  "4급(대리) 직원의 국내출장 일비는 하루에 얼마인가요?"
             └ "4급(대리) 직원의 국내출장 일비는 하루에 얼마인가요?"
3갈래 예 : G02  "4급 직원이 광역시로 출장 갈 때 숙박비 상한은 얼마인가요?"
             └ "4급 직원이 광역시로 출장 갈 때 숙박비 상한은 얼마인가요?"
             └ "4급 직원이 광역시로 출장 갈 때 숙박비 상한은 얼마인가요? 숙박비 호텔 1박"
             └ "4급 직원이 광역시로 출장 숙박비 상한은 얼마인가요?"


In [8]:
import importlib
from app.core import config
from app.integrations.factory import get_embedder
from app.rag import retriever, store

importlib.reload(config)  # config.py 를 고친 뒤에도 커널에 남은 옛 Settings 대신 새 버전을 쓰도록
importlib.reload(retriever)  # retriever 도 새 config 를 보도록 다시 불러옴
settings = config.get_settings()

EVAL_USER = {'id': '2011-0003', 'name': '한서영', 'clearance': '대외비', 'dept_id': 'HR'}
QUESTION = '출장비 얼마까지 나오나요?'

with session_scope() as s:
    embedder = get_embedder()
    (before, seen) = (0, set())
    for one in retriever.expand_queries(QUESTION, settings.multi_query_n):
        rows = store.vector_search(s, embedder.embed_query(one), limit=settings.retrieval_top_k)
        before += len(rows)
        seen.update((row['chunk_id'] for row in rows))
    result = retriever.LiveRetriever().search(question=QUESTION, user=EVAL_USER, session=s)
print(f"[1단계] {result.steps[0]['name']}   ({result.steps[0]['time']})")
print(f'중복 제거 전 : {before}건  →  후 : {len(seen)}건')

for (i, one) in enumerate(result.sources, 1):
    print(f"  {i}  {one['doc_id']}  {one['version']}  {one['locator']}   score {one['score']:.2f}")

14:29:36 INFO     sentence_transformers.base.model: No device provided, using cpu
14:29:36 INFO     sentence_transformers.base.model: Loading SentenceTransformer model from models\bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

14:29:38 INFO     app.integrations.local_embed: 로컬 임베딩 모델 준비: models\bge-m3/1024 차원


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[1단계] 질의 재작성 - Multi-Query 2건 생성   (0.0s)
중복 제거 전 : 24건  →  후 : 18건
  1  DOC-HR-014  v1.1  제1장 제2조(정의)   score 0.72
  2  DOC-HR-014  v2.0  제3장 제2절 제14조(숙박비) · 별표1   score 0.71
  3  DOC-HR-014  v1.1  제1장 제4조(여비의 종류) · 별표1   score 0.71


In [9]:
from sqlalchemy import select
from app.db.session import session_scope
from app.models.document import Chunk
from app.rag import embedder, store

NEEDLES = ['100,000', 'KTX']

with session_scope() as s:
    for needle in NEEDLES:
        locs = s.execute(select(Chunk.locator).where(Chunk.text.like(f'%{needle}%')).order_by(Chunk.id)).scalars().all()
        print(f"[글자] LIKE '%{needle}%'".ljust(23) + f'→  {len(locs)}건')
        for loc in locs[:3]:
            print(f'         · {loc}')
    rows = store.vector_search(s, embedder.embed_query(NEEDLES[0]), limit=3)
print(f'[벡터] "{NEEDLES[0]}" 상위 3건')
for (rank, row) in enumerate(rows, 1):
    print(f"         {rank} {row['locator']}   score {row['score']:.3f}")


[글자] LIKE '%100,000%'  →  1건
         · 표2
[글자] LIKE '%KTX%'      →  3건
         · 제3장 제3절 제18조(철도)
         · 표4
         · 제3장 제10조(교통비)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[벡터] "100,000" 상위 3건
         1 표2   score 0.451
         2 표1   score 0.442
         3 표2   score 0.438


In [12]:
import importlib

from app.rag import store

importlib.reload(store)

print('숙박비는 1박당 70,000원')
print(store._tokenize('숙박비는 1박당 70,000원'))

숙박비는 1박당 70,000원
['숙박비는', '1박당', '70', '000원']


In [14]:
from app.db.session import session_scope
from app.rag import embedder, store

QUESTION='1~2급 직원의 숙박비 상한은 얼마?'

with session_scope() as s:
   vec=store.vector_search(s, embedder.embed_query(QUESTION),limit=3)
   key=store.keyword_search(s, QUESTION, limit=3)

print('질문:',QUESTION)
print()
print('[벡터]'.ljust(40)+'[키워드 BM25]')
for rank, (v,k) in enumerate(zip(vec,key),1):
   left=f'{rank} {v['locator']} {v['score']:.3f}'
   print(f'{left:<40} {rank} {k['locator']} {k['score']:.3f}')

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

질문: 1~2급 직원의 숙박비 상한은 얼마?

[벡터]                                    [키워드 BM25]
1 표2 0.710                               1 제3장 제11조(숙박비) · 별표1 1.000
2 제3장 제2절 제14조(숙박비) · 별표1 0.689          2 표2 0.860
3 표2 0.687                               3 표2 0.855


In [15]:
import importlib

from app.db.session import session_scope
from app.rag import store

importlib.reload(store)

QUESTION='해외 출장 시 비즈니스석 탑승 기준은?' # 문서에 없는 질문

In [17]:
import importlib
from app.db.session import session_scope
from app.rag import retriever, store

importlib.reload(store)
importlib.reload(retriever)

QUESTION = '1~2급 직원의 숙박비 상한은 얼마인가요?'
USER = {'name': '김민준', 'dept': 'INFRA2', 'clearance': '일반'}
with session_scope() as s:
    found = retriever.LiveRetriever().search(question=QUESTION, user=USER, session=s)
for (i, step) in enumerate(found.steps, 1):
    print(f"[{i}단계] {step['name']}  ({step['time']})   state={step['state']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[1단계] 질의 재작성 - Multi-Query 2건 생성  (0.0s)   state=ok
[2단계] 벡터 + 키워드 하이브리드 검색 · 벡터 14건 · 키워드 12건 -> 후보 18  (0.6s)   state=ok
